# Does `block_network=True` hold inside a Fabric notebook?

Eleven checks. Each prints PASS or FAIL and the run prints a total at the end, so
a partial pass is visible rather than looking like success.

What is being verified: with `block_network=True`, code running in the RLM
worker cannot open a connection off the machine, while everything that should
keep working still does - the worker starts, loopback is fine, the main LM is
unaffected, and local files still read.

Why this needs checking on Fabric specifically. The worker is a subprocess of
the notebook kernel inside a Synapse executor. The guard is delivered through
the child's environment, so it depends on env inheritance behaving the same way
there as on a laptop. That is the assumption under test.

**Result: 11 of 11 passed on a Fabric capacity, 2026-08-03.** Both LM paths
were exercised - an OpenRouter key and the built-in `FabricLM` - and check 11
confirmed the worker was still sealed during a run whose LM was reachable.
Check 6 passed too, so the workspace does have outbound access and checks 2 and
3 were falsifiable rather than passing by default.

**Install the released package first** (restart the session after installing):

```
%pip install "fabric-rlm==0.5.0"
```


In [ ]:
%pip install -q "fabric-rlm==0.5.0"

In [ ]:
import fabric_rlm, socket, os, sys
print("fabric_rlm", fabric_rlm.__version__)
print("python", sys.version.split()[0])
from fabric_rlm import RLM
from fabric_rlm.netguard import ENV_FLAG

results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (("  | " + str(detail)[:120]) if detail else ""))

class ScriptedLM:
    """Local stand-in for an LM: no API key needed for checks 1-8."""
    def __init__(self, turns): self.turns, self.i = list(turns), 0
    def __call__(self, messages=None, prompt=None, **kw):
        if self.i >= len(self.turns): return ["SUBMIT(answer='done')"]
        t = self.turns[self.i]; self.i += 1; return [t]

class LiveCheckedScriptedLM(ScriptedLM):
    """Reach the real parent LM once, then emit deterministic worker code."""
    def __init__(self, live_lm, turns):
        super().__init__(turns)
        self.live_lm = live_lm
        self.live_checked = False
    def __call__(self, messages=None, prompt=None, **kw):
        if not self.live_checked:
            response = self.live_lm("Reply with exactly OK.")
            if not response:
                raise RuntimeError("FabricLM returned no response")
            self.live_checked = True
        return super().__call__(messages=messages, prompt=prompt, **kw)

def code(body): return "```python\n" + body + "\n```"

def run_one(body, lm=None, **kw):
    lm = lm or ScriptedLM([code(body), code("SUBMIT(answer='done')")])
    return RLM.task(task="t", outputs=["answer"], lm=lm, max_turns=4,
                    timeout=120, **kw).run()


In [ ]:
# 1. The worker starts at all with the guard on.
# This is the check that failed first time round: denying socket construction
# outright killed the worker before turn one, because asyncio's event loop
# builds its self-pipe with socket.socketpair().
r = run_one("print('worker alive')", block_network=True)
out = (r.trajectory.turns[0].stdout or "")
check("1. worker starts and runs a turn with block_network=True",
      r.submitted and "worker alive" in out, r.trajectory.turns[0].error or out.strip())


In [ ]:
# 2. A remote connection is refused.
# security is disabled here on purpose: SecurityPolicy's static denylist would
# reject socket.create_connection at the source level, so with it on this would
# pass whether or not netguard works. Testing the wrong layer proves nothing.
from fabric_rlm.security import SecurityPolicy

body = '''
import socket
try:
    socket.create_connection(("93.184.216.34", 80), timeout=5)
    print("NOT BLOCKED")
except OSError as e:
    print("blocked:", "egress is blocked" in str(e))
'''
r = run_one(body, block_network=True, security=SecurityPolicy.disabled())
out = (r.trajectory.turns[0].stdout or "")
check("2. remote connect refused in the worker", "blocked: True" in out, out.strip())


In [ ]:
# 3. A library-wrapped fetch is refused - the real leak vector.
# The model's code names no denied symbol here, which is exactly why a
# source-level denylist cannot see it.
body = '''
import urllib.request as u
try:
    u.urlopen("https://huggingface.co", timeout=5)
    print("NOT BLOCKED")
except Exception as e:
    print("blocked:", "egress is blocked" in str(e))
'''
r = run_one(body, block_network=True, security=SecurityPolicy.disabled())
out = (r.trajectory.turns[0].stdout or "")
check("3. urllib fetch refused (library-wrapped call)", "blocked: True" in out, out.strip())


In [ ]:
# 4. Loopback still works. Anything local must keep functioning, or the
# guard breaks legitimate notebook code.
body = '''
import socket
s = socket.socket(); s.bind(("127.0.0.1", 0)); s.listen(1)
port = s.getsockname()[1]
c = socket.create_connection(("127.0.0.1", port), timeout=5)
print("loopback ok"); c.close(); s.close()
'''
r = run_one(body, block_network=True, security=SecurityPolicy.disabled())
out = (r.trajectory.turns[0].stdout or "")
check("4. loopback connection still allowed", "loopback ok" in out, out.strip())


In [ ]:
# 5. Off by default. The flag must not be in the child environment
# unless it was asked for.
r = run_one("import os; print('flag=', os.environ.get(%r))" % ENV_FLAG)
out = (r.trajectory.turns[0].stdout or "")
check("5. off by default (no flag in worker env)", "flag= None" in out, out.strip())


In [ ]:
# 6. Egress still works in the worker when NOT asked for. Without this,
# 'blocked' in check 2 might just mean Fabric has no outbound access at all,
# and the whole notebook would prove nothing.
body = '''
import socket
try:
    socket.create_connection(("1.1.1.1", 53), timeout=8)
    print("egress available")
except OSError as e:
    print("no egress:", type(e).__name__, str(e)[:60])
'''
r = run_one(body, security=SecurityPolicy.disabled())
out = (r.trajectory.turns[0].stdout or "")
if "egress available" in out:
    check("6. control: worker CAN connect out when block_network is off", True)
else:
    check("6. control: worker CAN connect out when block_network is off", False,
          "This workspace may have no outbound access, which makes checks 2 and 3 "
          "unfalsifiable here. Result: " + out.strip())


In [ ]:
# 7. Local file reads still work. The guard is about the network, and
# should not touch the filesystem or Lakehouse access.
import tempfile, pathlib
p = pathlib.Path(tempfile.gettempdir()) / "bn_probe.txt"
p.write_text("local data intact", encoding="utf-8")
r = run_one("from pathlib import Path; print(Path(r'%s').read_text())" % p,
            block_network=True)
out = (r.trajectory.turns[0].stdout or "")
check("7. local file reads unaffected", "local data intact" in out, out.strip())


In [ ]:
# 8. An explicit sub_lm is rejected up front rather than failing mid-run.
# Sub-LM calls are made from inside the worker, which now has no egress.
try:
    RLM.task(task="t", outputs=["a"], lm=ScriptedLM([]),
             block_network=True, sub_lm={"model": "x"})
    check("8. explicit sub_lm rejected at construction", False, "no error raised")
except ValueError as e:
    check("8. explicit sub_lm rejected at construction",
          "cannot be combined" in str(e), str(e)[:100])


In [ ]:
# 9. The real thing: a live LM call from the PARENT still works with the
# worker sealed. This is the property that makes the feature usable - the model
# is reachable, the sandbox is not.
#
# Read the key from the notebook environment. Skipped, not failed, if absent.
import os
KEY = os.environ.get("OPENROUTER_API_KEY", "")

if not KEY:
    print("SKIP  9. live LM check - set OPENROUTER_API_KEY to run it")
else:
    # Caught, not raised: an expired key must not abort the notebook and cost
    # you the summary for checks 1-8. A credentials failure is reported as a
    # skip because it says nothing about whether the guard works.
    try:
        r = RLM.task(
            task="Compute 6 * 7 in Python and submit it as the answer.",
            outputs=["answer"],
            lm={"model": "openrouter/minimax/minimax-m3", "api_key": KEY,
                "api_base": "https://openrouter.ai/api/v1", "timeout": 300},
            max_turns=6, timeout=300, block_network=True,
        ).run()
        got = str((r.payload or {}).get("answer", ""))
        check("9. live LM call from parent works while worker is sealed",
              r.submitted and "42" in got, got[:80])
    except Exception as e:
        name = type(e).__name__
        if "Auth" in name or "401" in str(e) or "RateLimit" in name:
            print("SKIP  9. live LM check - credentials problem, not a guard "
                  "failure: %s" % str(e)[:110])
        else:
            check("9. live LM call from parent works while worker is sealed",
                  False, "%s: %s" % (name, str(e)[:110]))


In [ ]:
# 10. Same again with Fabric's built-in LM.
#
# This is the one that matters most on Fabric, and it is not implied by check 9.
# FabricLM authenticates through the workspace rather than an API key, so it
# uses a different code path to reach the endpoint. Both calls happen in the
# parent process, so both should be unaffected by a sealed worker - but "should
# be" is the thing being tested.
try:
    from fabric_rlm import FabricLM
    r = RLM.task(
        task="Compute 6 * 7 in Python and submit it as the answer.",
        outputs=["answer"],
        lm=FabricLM("gpt-5.1"),
        max_turns=6, timeout=300, block_network=True,
    ).run()
    got = str((r.payload or {}).get("answer", ""))
    check("10. FabricLM works while the worker is sealed",
          r.submitted and "42" in got, got[:80])
except Exception as e:
    msg = str(e)
    # Off Fabric there is no synapse module, which is not a guard failure. Skip
    # rather than fail, so a local dry run does not report a false problem.
    if isinstance(e, ImportError) or "synapse" in msg or "notebookutils" in msg:
        print("SKIP  10. FabricLM not available - run this on a Fabric capacity")
    else:
        check("10. FabricLM works while the worker is sealed", False,
              "%s: %s" % (type(e).__name__, msg[:110]))


In [ ]:
# 11. The guard is still on during a real LM run.
# Checks 9 and 10 prove the model is reachable. This proves that reachability
# did not come at the cost of the worker being sealed - i.e. the two are not
# in conflict, which is the whole premise of the feature.
from fabric_rlm.security import SecurityPolicy

probe = '''
import socket
try:
    socket.create_connection(("1.1.1.1", 53), timeout=5)
    print("WORKER CAN STILL REACH OUT")
except OSError as e:
    print("worker sealed:", "egress is blocked" in str(e))
'''
try:
    from fabric_rlm import FabricLM
    lm_for_probe = LiveCheckedScriptedLM(
        FabricLM("gpt-5.1"),
        [code(probe), code("SUBMIT(answer='done')")],
    )
except Exception:
    lm_for_probe = None

if lm_for_probe is None:
    print("SKIP  11. needs FabricLM")
else:
    try:
        r = run_one(probe, lm=lm_for_probe, block_network=True,
                    security=SecurityPolicy.disabled())
        out = (r.trajectory.turns[0].stdout or "")
        check("11. worker still sealed in a run whose LM is reachable",
              "worker sealed: True" in out, out.strip())
    except Exception as e:
        name = type(e).__name__
        if "Auth" in name or "401" in str(e) or "RateLimit" in name:
            print("SKIP  11. live LM check - credentials problem, not a guard "
                  "failure: %s" % str(e)[:110])
        else:
            check("11. worker still sealed in a run whose LM is reachable",
                  False, "%s: %s" % (name, str(e)[:110]))


In [ ]:
passed = sum(1 for _, ok in results if ok)
total = len(results)
print()
print("%d of %d checks passed" % (passed, total))
for name, ok in results:
    if not ok:
        print("  FAILED:", name)
if passed != total:
    print()
    print("Report the failing names back. A partial pass means the guard does not")
    print("hold in this environment, which is worth knowing before relying on it.")
assert passed == total, "Network guard verification failed"
